# 04 Modelo de inteligencia de negocios — ENSO
## Proyecto cambio climático | Dataset: Cambio_climatico.csv (1950–2026)

Este notebook documenta el modelo de BI sobre el dataset ENSO procesado:
esquema estrella, KPIs ejecutivos e insights para el dashboard.

# 1. Importación de librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.2f}'.format)

# 2. Carga del dataset

Se carga `enso_model_ready.csv`, resultado del pipeline ETL (Fase 1).

In [ ]:
df = pd.read_csv('../data/processed/enso_model_ready.csv')
print('Shape:', df.shape)
print('Periodo:', df['Anio'].min(), '-', df['Anio'].max())
print('Columnas:', df.columns.tolist())
df.head()

# 3. Descripción del dataset

In [ ]:
print(df.dtypes)
print()
df[['Temperatura_Pacifico_C','Anomalia_C','Duracion_Meses','Evento_Extremo']].describe()

# 4. Modelo dimensional — Esquema Estrella

El modelo sigue un **esquema estrella** con una tabla de hechos y dos dimensiones:

| Tabla | Tipo | Filas | Descripción |
|---|---|---|---|
| FACT_ENSO | Hechos | 914 | 1 registro mensual con todas las métricas |
| DIM_TIEMPO | Dimensión | 914 | Fecha, Año, Mes, Trimestre, Década |
| DIM_FASE | Dimensión | 5 | Fase del evento + Intensidad |

Las métricas en FACT_ENSO son: Temperatura_Pacifico_C, Anomalia_C, Duracion_Meses y Evento_Extremo.

In [ ]:
dim_tiempo = (df[['Fecha','Anio','Mes','Trimestre','Decada']]
              .drop_duplicates().sort_values('Fecha').reset_index(drop=True))
dim_tiempo.insert(0, 'id_tiempo', range(1, len(dim_tiempo)+1))

dim_fase = (df[['Fase_Evento','Intensidad_Evento']]
            .drop_duplicates().sort_values('Fase_Evento').reset_index(drop=True))
dim_fase.insert(0, 'id_fase', range(1, len(dim_fase)+1))

fact_cols = ['id_tiempo','id_fase','Temperatura_Pacifico_C',
             'Temperatura_Ajustada_C','Anomalia_C','Duracion_Meses','Evento_Extremo']
fact_enso = (df.merge(dim_tiempo[['Fecha','id_tiempo']], on='Fecha')
               .merge(dim_fase[['Fase_Evento','Intensidad_Evento','id_fase']],
                      on=['Fase_Evento','Intensidad_Evento'])
               [fact_cols])

print('DIM_TIEMPO:', dim_tiempo.shape)
print(dim_tiempo.head(3).to_string())
print()
print('DIM_FASE:', dim_fase.shape)
print(dim_fase.to_string())
print()
print('FACT_ENSO:', fact_enso.shape)
print(fact_enso.head(3).to_string())

# 5. KPIs ejecutivos

In [ ]:
anomalia_avg = round(df['Anomalia_C'].mean(), 4)
anomalia_max = round(df['Anomalia_C'].max(), 2)
anomaila_min = round(df['Anomalia_C'].min(), 2)
eventos_ext  = int(df['Evento_Extremo'].sum())
eventos_pct  = round(df['Evento_Extremo'].mean()*100, 2)
decada_max   = df.groupby('Decada')['Evento_Extremo'].sum().idxmax()
fase_frec    = df['Fase_Evento'].value_counts()
dur_max      = int(df['Duracion_Meses'].max())

print('='*55)
print(f'  ANOMALÍA PROMEDIO GLOBAL  :  {anomalia_avg} °C')
print(f'  Anomalía máxima / mínima  :  +{anomalia_max} / {anomalia_max} °C')
print()
print(f'  EVENTOS EXTREMOS TOTALES  :  {eventos_ext}  ({eventos_pct}% de registros)')
print(f'  Década más activa         :  {decada_max}')
print()
print(f'  FASE MÁS FRECUENTE        :  {fase_frec.idxmax()} ({fase_frec.max()} meses)')
print(f'  Evento más largo          :  {dur_max} meses')
print('='*55)

# 6. Visualizaciones de apoyo al dashboard

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Dashboard BI — ENSO 1950-2026', fontsize=14)

anual = df.groupby('Anio')['Anomalia_C'].mean()
axes[0,0].plot(anual.index, anual.values, color='steelblue', linewidth=1.2)
axes[0,0].axhline(0, color='gray', linestyle='--', linewidth=0.8)
axes[0,0].fill_between(anual.index, anual.values, 0,
    where=(anual.values>0), alpha=0.3, color='tomato', label='El Niño')
axes[0,0].fill_between(anual.index, anual.values, 0,
    where=(anual.values<0), alpha=0.3, color='cornflowerblue', label='La Niña')
axes[0,0].set_title('Anomalía de temperatura 1950-2026')
axes[0,0].set_ylabel('Anomalía (°C)')
axes[0,0].legend()

ext_dec = df.groupby('Decada')['Evento_Extremo'].sum().sort_index()
axes[0,1].bar(ext_dec.index, ext_dec.values, color='tomato')
axes[0,1].set_title('Eventos extremos por década')
axes[0,1].set_ylabel('Número de eventos')
axes[0,1].tick_params(axis='x', rotation=30)

fase_c = df['Fase_Evento'].value_counts()
axes[1,0].pie(fase_c.values, labels=fase_c.index, autopct='%1.1f%%',
              colors=['tomato','cornflowerblue','lightgray'])
axes[1,0].set_title('Distribución de fases ENSO')

int_fase = df.groupby(['Fase_Evento','Intensidad_Evento']).size().unstack(fill_value=0)
int_fase.plot(kind='bar', ax=axes[1,1], colormap='Set2')
axes[1,1].set_title('Intensidad por fase')
axes[1,1].set_ylabel('Meses')
axes[1,1].tick_params(axis='x', rotation=15)
axes[1,1].legend(title='Intensidad', fontsize=8)

plt.tight_layout()
plt.savefig('../reports/bi_enso_dashboard.png', dpi=120, bbox_inches='tight')
plt.show()

# 7. Insights del dashboard

## Insight 1 — Tendencia de intensificación
Los eventos extremos se concentran en las décadas 1980s, 1990s y 2010s,
con eventos históricos como El Niño 1982-83, 1997-98 y 2015-16.

## Insight 2 — Fase Neutral predomina
El 44% de los meses registrados corresponden a fase Neutral, el 31% a La Niña
y el 25% a El Niño. La irregularidad es una característica estructural del ciclo.

## Insight 3 — Duración como indicador de impacto
Los eventos más largos (>30 meses) coinciden con los de mayor anomalía.
La duración es el predictor más importante para identificar eventos extremos.

## Insight 4 — Colombia como región de alta exposición
Los años de impacto severo en Colombia coinciden con los picos de anomalía.
El Niño fuerte genera sequías; La Niña intensa genera inundaciones.

# 8. Reglas de negocio del modelo

In [ ]:
reglas = {
    'RN-01': '1 registro mensual por fecha (sin duplicados)',
    'RN-02': 'Anomalia_C = temperatura observada menos media histórica (°C)',
    'RN-03': 'Evento_Extremo = 1 si |Anomalia_C| > 1.5°C',
    'RN-04': 'Fase_Evento: El Nino, La Nina, Neutral',
    'RN-05': 'Duracion_Meses = meses consecutivos del evento activo',
    'RN-06': 'Periodo de análisis: 1950-2026 (914 registros mensuales)',
    'RN-07': 'Anomalia_C es la métrica principal del sistema ENSO',
}
for k, v in reglas.items():
    print(f'  {k}: {v}')